In [5]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize
from scipy.stats import chi2
import warnings
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['axes.unicode_minus'] = False


def preprocess_data(df, core_vars, target_var, threshold=0.7):
    
    
    df = df.dropna(subset=core_vars + [target_var])
    
    
    for var in core_vars:
        if df[var].dtype in [np.float64, np.int64]:
            mean = df[var].mean()
            std = df[var].std()
            df = df[(df[var] >= mean - 3*std) & (df[var] <= mean + 3*std)]
    
    
    def drop_collinear(df, vars_list, threshold):
        corr_matrix = df[vars_list].corr()
        drop_cols = []
        for i in range(len(corr_matrix.columns)):
            for j in range(i):
                if abs(corr_matrix.iloc[i, j]) > threshold:
                    colname = corr_matrix.columns[i]
                    if colname not in drop_cols:
                        drop_cols.append(colname)
        return df.drop(columns=drop_cols), drop_cols
    
    df_cleaned, dropped_cols = drop_collinear(df, core_vars, threshold)
    if dropped_cols:
        print(f"⚠️ Remove variables due to collinearity：{dropped_cols}")
        core_vars_updated = [v for v in core_vars if v not in dropped_cols]
    else:
        core_vars_updated = core_vars.copy()
    
    
    for var in core_vars_updated:
        if df_cleaned[var].dtype in [np.float64, np.int64]:
            df_cleaned[var] = (df_cleaned[var] - df_cleaned[var].mean()) / df_cleaned[var].std()
    
    
    
    global nest_structure
    df_cleaned['nest_temp'] = df_cleaned[target_var].map(
        {c: 'low_risk' if c in nest_structure['low_risk'] else 'high_risk' for c in df_cleaned[target_var].unique()}
    )
    low_risk_df = df_cleaned[df_cleaned['nest_temp'] == 'low_risk']
    high_risk_df = df_cleaned[df_cleaned['nest_temp'] == 'high_risk']
    min_samples = min(len(low_risk_df), len(high_risk_df))
    if min_samples > 0:
        df_balanced = pd.concat([
            low_risk_df.sample(min_samples, random_state=42),
            high_risk_df.sample(min_samples, random_state=42)
        ])
    else:
        df_balanced = df_cleaned.copy()
    
    return df_balanced, core_vars_updated



data_path = "EB-motor crashes (NAIS).xlsx"
df = pd.read_excel(data_path, engine='openpyxl')


target_var = "Accident classification"  
core_vars = [
    "Vehicle type",          
    "Driver gender",         
    "Driver age",            
    "Estimated vehicle speed(km/h)", 
    "Weather",               
    "Driver illegal fault",  
    "Vehicle maneuver"       
]


missing_vars = [v for v in core_vars + [target_var] if v not in df.columns]
if missing_vars:
    print("⚠️ The actual list of column names in the data：")
    print(df.columns.tolist())
    raise ValueError(f"❌ The following variables are missing from the data：{missing_vars}，Please check the spelling of the variable name!")


target_cats = sorted(df[target_var].unique())
if len(target_cats) < 3:
    raise ValueError(f"❌ The nested Logit model requires at least 3 accident classifications! Currently, only [insert number] have been found：{len(target_cats)}个")

nest_structure = {
    'low_risk': [target_cats[0]],       
    'high_risk': [target_cats[1], target_cats[2]]  
}


df_model, core_vars = preprocess_data(df, core_vars, target_var, threshold=0.7)


nest_mapping = {}
for nest, cats in nest_structure.items():
    for cat in cats:
        nest_mapping[cat] = nest
df_model['nest'] = df_model[target_var].map(nest_mapping)


df_model['target_encoded'] = df_model[target_var].map({c:i for i,c in enumerate(target_cats)})
nest_ids = {nest:i for i,nest in enumerate(nest_structure.keys())}
df_model['nest_encoded'] = df_model['nest'].map(nest_ids)

# 数据信息输出
print("=== Nested Logit Model Data Information ===")
print(f"📊 Row data：{len(df)}  → Processed data：{len(df_model)} ")
print(f"🚨 Accident classication：{target_cats}")
print(f"🏷️ NLM：{nest_structure}")
print(f"🔑 Core variable：{core_vars}")


def nested_logit_neg_log_likelihood(params, X, y, nest_ids, n_nests, n_choices_per_nest):
    
    n_params = X.shape[1]
    n_nest_params = n_nests  
    
    
    iv_params = params[:n_nest_params]  
    beta_within = params[n_nest_params : n_nest_params + sum(n_choices_per_nest)*n_params].reshape(sum(n_choices_per_nest), n_params)
    beta_between = params[n_nest_params + sum(n_choices_per_nest)*n_params:].reshape(n_nests, n_params)
    
    
    iv_params = np.exp(iv_params) + 1e-10  
    
    log_likelihood = 0
    
    util_within = np.dot(X, beta_within.T)
    
    util_within = util_within - np.max(util_within, axis=1, keepdims=True)
    
    
    iv_all = np.zeros((len(X), n_nests))
    start_idx = 0
    for nest in range(n_nests):
        n_choices = n_choices_per_nest[nest]
        nest_util = util_within[:, start_idx:start_idx+n_choices]
        exp_util = np.exp(iv_params[nest] * nest_util)
        exp_util_sum = np.sum(exp_util, axis=1) + 1e-10  
        nest_iv = (1/iv_params[nest]) * np.log(exp_util_sum)
        iv_all[:, nest] = nest_iv
        start_idx += n_choices
    
    
    util_between = np.dot(X, beta_between.T) + iv_all
    util_between = util_between - np.max(util_between, axis=1, keepdims=True)
    
    
    exp_between = np.exp(util_between)
    prob_between = exp_between / (np.sum(exp_between, axis=1, keepdims=True) + 1e-10)
    
    
    total_prob = np.zeros(len(X))
    for i in range(len(X)):
        nest_idx = nest_ids[i]
        choice_idx = y[i]
        
        
        start_idx_nest = sum(n_choices_per_nest[:nest_idx])
        end_idx_nest = start_idx_nest + n_choices_per_nest[nest_idx]
        
        
        nest_util_i = util_within[i, start_idx_nest:end_idx_nest]
        exp_within_i = np.exp(iv_params[nest_idx] * nest_util_i)
        prob_within_i = exp_within_i / (np.sum(exp_within_i) + 1e-10)
        
        
        within_idx = choice_idx - start_idx_nest
        if within_idx < 0 or within_idx >= len(prob_within_i):
            total_prob[i] = 1e-10  
        else:
            total_prob[i] = prob_between[i, nest_idx] * prob_within_i[within_idx] + 1e-10
    
    
    log_likelihood = np.sum(np.log(total_prob))
    
    return -log_likelihood

def fit_nested_logit(X, y, nest_ids, nest_structure, target_cats):
    
    n_nests = len(nest_structure)
    n_choices_per_nest = [len(cats) for cats in nest_structure.values()]
    n_params = X.shape[1]
    
    
    n_nest_params = n_nests
    
    n_iv_params = n_nest_params
    n_within_params = sum(n_choices_per_nest) * n_params
    n_between_params = n_nests * n_params
    total_params = n_iv_params + n_within_params + n_between_params
    
    
    init_iv = np.log(np.ones(n_iv_params))  
    
    init_within = np.random.normal(0, 0.01, n_within_params)
    init_between = np.random.normal(0, 0.01, n_between_params)
    init_params = np.concatenate([init_iv, init_within, init_between])
    
    
    bounds = []
    
    bounds += [(None, np.log(10)) for _ in range(n_iv_params)]
    
    bounds += [(-5, 5) for _ in range(n_within_params + n_between_params)]
    
    
    result = minimize(
        nested_logit_neg_log_likelihood,
        init_params,
        args=(X, y, nest_ids, n_nests, n_choices_per_nest),
        method='L-BFGS-B',  
        bounds=bounds,      
        options={
            'gtol': 1e-6,       
            'maxiter': 10000,   
            'disp': True,       
            'maxfun': 50000     
        }
    )
    
    
    iv_params = np.exp(result.x[:n_nest_params]) + 1e-10  
    
    beta_within = result.x[n_nest_params : n_nest_params + n_within_params].reshape(sum(n_choices_per_nest), n_params)
    
    beta_between = result.x[n_nest_params + n_within_params:].reshape(n_nests, n_params)
    
    return {
        'iv_params': iv_params,
        'beta_within': beta_within,  
        'beta_between': beta_between, 
        'log_likelihood': -result.fun,
        'converged': result.success,
        'n_iter': result.nit,
        'message': result.message  
    }


X = np.hstack([np.ones((len(df_model), 1)), df_model[core_vars].values])
X = X.astype(np.float64)  
y = df_model['target_encoded'].values
nest_ids = df_model['nest_encoded'].values


print("\n=== Start fitting the NLM ===")
nested_results = fit_nested_logit(X, y, nest_ids, nest_structure, target_cats)


print("\n=== NLM fitting results ===")
print(f"✅ Model convergence state：{nested_results['converged']}")
print(f"📝 Solver Information：{nested_results['message']}")
print(f"📈 Log-likelihood value：{nested_results['log_likelihood']:.4f}")
print(f"🔄 Number of iterations：{nested_results['n_iter']}")


print("\n=== Inclusive Value, IV ===")
for i, (nest_name, _) in enumerate(nest_structure.items()):
    iv_val = nested_results['iv_params'][i]
    if iv_val < 0.8:
        iv_interpret = "Intra-layer selection with high correlation"
    elif iv_val < 1.0:
        iv_interpret = "Intra-layer moderate independence selection"
    else:
        iv_interpret = "Intra-layer selection is almost independent"
    print(f"📌 {nest_name} layer IV：{iv_val:.4f} → {iv_interpret}")


print("\n=== Core variable - Accident classification coefficient (within layer) ===")
coef_within_df = pd.DataFrame(
    nested_results['beta_within'],
    index=[f"Acciendt classication_{cat}" for cat in target_cats],
    columns=['Constant term'] + core_vars
)
print(coef_within_df.round(4))


print("\n=== Core variable - Nested layer coefficient (inter-layer) ===")
coef_between_df = pd.DataFrame(
    nested_results['beta_between'],
    index=list(nest_structure.keys()),
    columns=['Constant term'] + core_vars
)
print(coef_between_df.round(4))


print("\n=== Analysis of the Impact of Core Variables on Accident Classification ===")

var_impact = {}
for i, var in enumerate(core_vars):
    
    mean_within = np.mean(np.abs(coef_within_df[var].values))
    
    mean_between = np.mean(np.abs(coef_between_df[var].values))
    var_impact[var] = (mean_within + mean_between) / 2


sorted_impact = sorted(var_impact.items(), key=lambda x: x[1], reverse=True)
for idx, (var, impact) in enumerate(sorted_impact, 1):
    impact_level = 'Strong' if impact>0.5 else 'Middle' if impact>0.2 else 'Weak'
    print(f"{idx}. {var}：Influence impact = {impact:.4f}（{impact_level}）")


fig, ax = plt.subplots(figsize=(12, 7))
vars_list = [v[0] for v in sorted_impact]
impact_list = [v[1] for v in sorted_impact]


colors = ['#e74c3c' if i<3 else '#f39c12' if i<5 else '#3498db' for i in range(len(vars_list))]
bars = ax.bar(vars_list, impact_list, color=colors, alpha=0.8, edgecolor='black', linewidth=0.5)


for bar, val in zip(bars, impact_list):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 0.01,
            f"{val:.4f}", ha='center', va='bottom', fontsize=9, fontweight='bold')

# 优化图表样式
ax.set_xlabel('Core variable', fontsize=12, fontweight='bold')
ax.set_ylabel('Impact intensity (average absolute value of coefficients)', fontsize=12, fontweight='bold')
ax.set_title('The influence intensity of the core variables on accident classification', fontsize=14, fontweight='bold', pad=20)
ax.tick_params(axis='x', rotation=45, labelsize=10)
ax.tick_params(axis='y', labelsize=10)
ax.grid(axis='y', alpha=0.3, linestyle='--')

# 添加参考线
ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.7, label='Strong influence threshold（0.5）')
ax.axhline(y=0.2, color='orange', linestyle='--', alpha=0.7, label='Middle influence threshold（0.2）')
ax.legend(fontsize=10, loc='upper right')

plt.tight_layout()
plt.savefig('The influence intensity of the NLM_core variable.png', dpi=300, bbox_inches='tight')
plt.close()
print("\n📸 The core variable influence intensity graph has been saved.")

=== Nested Logit Model Data Information ===
📊 Row data：1026  → Processed data：640 
🚨 Accident classication：[np.int64(1), np.int64(2), np.int64(3)]
🏷️ NLM：{'low_risk': [np.int64(1)], 'high_risk': [np.int64(2), np.int64(3)]}
🔑 Core variable：['Vehicle type', 'Driver gender', 'Driver age', 'Estimated vehicle speed(km/h)', 'Weather', 'Driver illegal fault', 'Vehicle maneuver']

=== Start fitting the NLM ===

=== NLM fitting results ===
✅ Model convergence state：True
📝 Solver Information：CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
📈 Log-likelihood value：-425.9900
🔄 Number of iterations：52

=== Inclusive Value, IV ===
📌 low_risk layer IV：1.0000 → Intra-layer moderate independence selection
📌 high_risk layer IV：10.0000 → Intra-layer selection is almost independent

=== Core variable - Accident classification coefficient (within layer) ===
                         Constant term  Vehicle type  Driver gender  \
Acciendt classication_1         0.2176       -0.2152        -0.5871   
Accien